# Periodically-supported UIC60 rail — 3D brick Bloch + transient tools (r2)

In-house 3D finite-element tools (numpy + scipy.sparse only, no imported mesh, no
black-box FE) for the lateral dynamics of the Delkor-HAF-supported UIC60 / 60E1
rail. Everything below is **executed**; the plots and numbers are the real
outputs of the runs.

**Coordinates (SI):** `X` lateral, `Y` vertical, `Z` axial (wave propagation).
The drive point is a lateral (X) force at a rail-head node.

**Contents**
1. Shared cell generator & UIC60 section (`rail_cell.py`)
2. Tool A — Floquet-Bloch dispersion (`bloch_dispersion.py`)
3. Tool B — transient full-wave solver with ALID ends (`transient_fullwave.py`)
4. Validation suite (`validate.py`)
5. What the tools say about the two-peak hypothesis

> Run order for a fresh kernel: execute cells top to bottom. The heavy transient
> and full validation figures are produced by the standalone scripts and embedded
> here; the fast pieces (section, a Bloch band) are recomputed live.

In [ ]:
import numpy as np, scipy, matplotlib
from IPython.display import Image, display
import rail_cell as rc, bloch_dispersion as A, transient_fullwave as B
print("numpy", np.__version__, " scipy", scipy.__version__)

## 1. Shared cell generator & the UIC60 section

`rail_cell.py` auto-builds the 60E1 cross-section from published nominal
dimensions, quad-meshes it with **axis-aligned** rectangular elements (a
width-mapped grid's slanted columns brace the section and make it ~100× too
stiff in lateral bending — verified), and extrudes it into 8-node
**incompatible-modes ("C3D8I")** bricks (plain trilinear bricks shear-lock
catastrophically on this thin-walled section). The section is verified against
the published constants before anything downstream runs.

In [ ]:
n2, q, mi = rc.build_cross_section()
_ = rc.verify_section(n2, q)          # prints A / Ix / Iy / mass vs 60E1

In [ ]:
# the actual quad mesh that is used
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(3.6, 7))
for quad in q:
    p = np.vstack([n2[quad], n2[quad][0]])
    ax.fill(p[:,0]*1e3, p[:,1]*1e3, facecolor="0.85", edgecolor="0.4", lw=0.4)
ax.set_aspect("equal"); ax.set_xlabel("x (mm)"); ax.set_ylabel("y (mm)")
ax.set_title("UIC60 axis-aligned quad mesh (%d quads)" % len(q))
fig.tight_layout(); fig.savefig("figures/nb_cross_section.png", dpi=120); plt.close(fig)
Image("figures/nb_cross_section.png")

In [ ]:
# bending sanity: cantilever tip deflection vs Euler-Bernoulli EI theory
import scipy.sparse.linalg as spla
from rail_cell import assemble_brick, Cfg
Ltot, nez = 2.0, 40
zs = np.linspace(0, Ltot, nez+1)
K, Mc, ml, n3, Ncs, ndof = assemble_brick(n2, q, zs, Cfg)
fixed = np.where(n3[:,2] < 1e-9)[0]
fd = np.concatenate([[3*n,3*n+1,3*n+2] for n in fixed])
free = np.setdiff1d(np.arange(ndof), fd); tip = np.where(n3[:,2] > Ltot-1e-9)[0]
Kff = K[free][:,free].tocsc()
for ld, I, name in [(0, 512e-8, "lateral (Iy)"), (1, 3038e-8, "vertical (Ix)")]:
    f = np.zeros(ndof); [f.__setitem__(3*n+ld, 1000.0/len(tip)) for n in tip]
    u = np.zeros(ndof); u[free] = spla.spsolve(Kff, f[free])
    ratio = u[3*tip+ld].mean() / (1000.0*Ltot**3/(3*Cfg.E*I))
    print(f"cantilever {name}: FE/theory = {ratio:.3f}")

## 2. Tool A — Floquet-Bloch dispersion

One unit cell (length `L`), reduction `u_Rf = e^{iκ} u_Lf`, reduced Hermitian
eigenproblem solved for each κ ∈ [0, π]. Branches are tracked by
modal-assurance continuity and coloured by **torsion fraction** (0 = lateral
bending, 1 = torsion/roll). Run live for the **V2 (reduced)** variant:

In [ ]:
resV2 = A.run_bloch(variant="V2", s_lat=1.0, n_kappa=21, n_branch=20)
A.plot_bands(resV2, "figures/nb_bloch_V2.png",
             title="Bloch bands — V2 (reduced), s_lat=1.0")
Image("figures/nb_bloch_V2.png")

**V1 (full, plate mass)** — the 9 kg plate on its springs (lateral resonance
≈ 556 Hz) appears as flat, dispersionless resonance branches that the rising
rail-bending branch **veers around** (a local-resonance avoided crossing). This
is the structure V2 does *not* have.

In [ ]:
display(Image("figures/bloch_bands_V1.png"))

**Lateral-stiffness sweep (V2).** Lateral support stiffness is the primary
calibration parameter. The bending pinned-pinned pair (~425 Hz) barely moves
(node at the support → support-stiffness-independent); the torsion/flapping
branch (~710–750 Hz) is the support-sensitive one.

In [ ]:
for s in [0.5, 1.0, 2.0]:
    display(Image("figures/bloch_bands_V2_slat%.2f.png" % s))

## 3. Tool B — transient full-wave solver (explicit CD + ALID)

`M u'' + C u' + K u = f(t)` marched explicitly with lumped mass and `C = α(x)·M`
ALID absorbing layers at both ends. A short **live** run below (broadband
Gaussian pulse, lateral drive at the head) — the peaks printed are from the FFT
of the *simulated* velocity response:

In [ ]:
resB = B.run_transient(variant="V2", s_lat=1.0, n_phys=4, n_alid=3,
                       ne_z=4, T_sim=0.08, pulse="gauss")
B.plot_mobility(resB, "figures/nb_mobility_live.png",
                title="Point mobility (live short run) — V2")
Image("figures/nb_mobility_live.png")

Full-resolution point/transfer mobilities and the space-time field + its
2D-FFT dispersion (produced by `python transient_fullwave.py`):

In [ ]:
for f in ["figures/mobility_V2.png", "figures/signatures_V2.png",
          "figures/spacetime_V2.png"]:
    display(Image(f))

## 4. Validation suite

`validate.py` runs the checks and writes `figures/validation_report.txt`. The
report and its figures (produced by `python validate.py`):

In [ ]:
print(open("figures/validation_report.txt").read())

In [ ]:
for f in ["figures/val_dispersion_overlay.png", "figures/val_Lshift.png",
          "figures/val_Nperiods.png", "figures/val_V1_vs_V2.png"]:
    display(Image(f))

## 5. What the tools say about the two-peak hypothesis

**Hypothesis:** the two lateral point-mobility peaks (~525/557 Hz) come from an
avoided crossing between a torsional/roll mode and a lateral-bending
(pinned-pinned) mode of the periodically supported rail.

**Findings (this 3D model):**
- The *ingredients* behave exactly as posited: a lateral-bending branch that
  **folds to a pinned-pinned standing wave at the zone edge** (node at the
  support → support-stiffness-independent) and a **torsion/flapping branch whose
  frequency is set by the lateral support stiffness**.
- **But on the bare rail (V2) they do not meet near 525/557**: bending folds at
  ~425 Hz, torsion sits at ~710–750 Hz, so there is no in-band torsion–bending
  crossing, and the V2 head-drive mobility carries no in-band two-peak feature.
- **Adding the plate (V1) produces the two-peak feature** (~633/675 Hz in the
  transient; extra hybrid branches in the Bloch diagram) via a **local-resonance
  hybridisation** at the 556 Hz plate resonance — the r1 mechanism.
- **Absolute frequencies carry a documented section-geometry offset** (~110 Hz
  high): the auto-built staircase reproduces A/Ix/Iy to a few % but omits the
  web/head fillets that set the torsion branch. Matching the measured 525/557
  curve needs fillet-resolved geometry or calibration to the measured figure
  (requested, not fabricated).

See `REPORT.md` for the full account.